<a href="https://colab.research.google.com/github/azcsprof/ASU-CSE475-SS25/blob/Unit-2-Lab-2/Random_Forest_Hyperparameter_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
import ipywidgets as widgets
from IPython.display import display, clear_output

# Load and prepare data
df = pd.read_csv("diabetes.csv")
X = df.drop("Outcome", axis=1)
y = df["Outcome"]

# Treat 0s as missing in key columns
cols_with_zeros = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
X[cols_with_zeros] = X[cols_with_zeros].replace(0, np.nan)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1)

# Widget controls
n_estimators = widgets.IntSlider(min=100, max=500, step=50, value=300, description='n_estimators')
max_depth = widgets.IntSlider(min=4, max=20, step=2, value=10, description='max_depth')
min_split = widgets.IntSlider(min=2, max=12, step=1, value=5, description='min_split')
min_leaf = widgets.IntSlider(min=1, max=5, step=1, value=2, description='min_leaf')
threshold = widgets.FloatSlider(min=0.1, max=0.9, step=0.01, value=0.5, description='Threshold')
k_folds = widgets.IntSlider(min=2, max=10, step=1, value=5, description='k_folds')

# Interactive model function
def update_model(n_estimators, max_depth, min_split, min_leaf, threshold, k_folds):
    clear_output(wait=True)

    # Create pipeline
    pipe = Pipeline([
        ('imputer', KNNImputer(n_neighbors=5)),
        ('scaler', StandardScaler()),
        ('rf', RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_split,
            min_samples_leaf=min_leaf,
            max_features='sqrt',
            class_weight='balanced',
            criterion='gini',
            random_state=1
        ))
    ])

    # Train and predict
    pipe.fit(X_train, y_train)
    probs = pipe.predict_proba(X_test)[:, 1]
    preds = (probs >= threshold).astype(int)

    # Evaluate
    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds)
    rec = recall_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    auc_score = roc_auc_score(y_test, probs)

    # Cross-validation
    skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=100)
    cv_scores = cross_val_score(pipe, X, y, cv=skf)
    cv_mean = np.mean(cv_scores) * 100

    # Output metrics
    print(f"Accuracy:  {acc:.4f} {'✔️' if acc >= 0.75 else '❌'}")
    print(f"Precision: {prec:.4f} {'✔️' if prec >= 0.75 else '❌'}")
    print(f"Recall:    {rec:.4f} {'✔️' if rec >= 0.75 else '❌'}")
    print(f"F1 Score:  {f1:.4f} {'✔️' if f1 >= 0.75 else '❌'}")
    print(f"ROC AUC:   {auc_score:.4f} {'✔️' if auc_score >= 0.80 else '❌'}")
    print(f"CV Accuracy ({k_folds}-fold): {cv_mean:.2f}% {'✔️' if cv_mean >= 72.0 else '❌'}")

    # 🔍 ROC Curve
    fpr, tpr, _ = roc_curve(y_test, probs)
    plt.figure(figsize=(6, 4))
    plt.plot(fpr, tpr, label=f'AUC = {auc_score:.2f}', color='darkorange')
    plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve')
    plt.legend()
    plt.grid(True)
    plt.show()

# Display
ui = widgets.VBox([n_estimators, max_depth, min_split, min_leaf, threshold, k_folds])
out = widgets.interactive_output(update_model, {
    'n_estimators': n_estimators,
    'max_depth': max_depth,
    'min_split': min_split,
    'min_leaf': min_leaf,
    'threshold': threshold,
    'k_folds': k_folds
})
display(ui, out)